# Note 6: The Multifrontal Method

**Goal:** Understand how the multifrontal method combines everything from Notes 1-5 into the state-of-the-art algorithm for sparse symmetric factorization — the algorithm implemented by rmumps.

## The Big Idea

The multifrontal method organizes sparse factorization as a **sequence of dense operations on small matrices** (frontal matrices), structured by the **elimination tree**.

Instead of working on the full sparse matrix with scattered operations, we:
1. Walk up the elimination tree from leaves to root
2. At each supernode, assemble a **dense frontal matrix** from the original entries + contributions from children
3. Partially factor it (eliminate the supernode's columns)
4. Pass the unfactored remainder (the **contribution block**) to the parent

This is brilliant because:
- Dense operations on frontal matrices use **BLAS Level-3** (fast!)
- The elimination tree reveals **parallelism** (independent branches)
- Fill-in is confined to frontal matrices — no global scatter/gather

## A Worked Example

Let's trace the multifrontal method on a small matrix, step by step.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

In [ ]:
# A small sparse symmetric matrix with known structure
#
# Sparsity pattern (symmetric, showing lower triangle):
#     0  1  2  3  4  5
# 0 [ x              ]
# 1 [    x            ]
# 2 [ x  x  x        ]
# 3 [       x  x      ]
# 4 [    x     x  x   ]
# 5 [          x  x  x]
#
# Elimination tree:   0 → 2 → 3 → 5
#                     1 → 2    4 → 5
#                              3 → 5

A = np.array([
    [ 4.0,  0.0,  1.0,  0.0,  0.0,  0.0],
    [ 0.0,  3.0,  1.0,  0.0,  1.0,  0.0],
    [ 1.0,  1.0,  5.0,  2.0,  0.0,  0.0],
    [ 0.0,  0.0,  2.0,  6.0,  1.0,  1.0],
    [ 0.0,  1.0,  0.0,  1.0,  4.0,  2.0],
    [ 0.0,  0.0,  0.0,  1.0,  2.0,  7.0]
])

print("A =")
print(A)
print(f"\nNonzeros: {np.count_nonzero(A)}")

# Verify it's SPD (so simple Cholesky works for comparison)
eigvals = np.linalg.eigvalsh(A)
print(f"Eigenvalues: {eigvals}")
print(f"Positive definite: {all(eigvals > 0)}")

In [ ]:
# The elimination tree for this matrix
# (computed from the sparsity pattern)

parent = [2, 2, 3, 5, 5, -1]  # parent[j] = parent of node j

# Supernodes (for this small example, each column is its own supernode)
# In practice, columns 0,1 could be grouped since they share parent 2
supernodes = [[0], [1], [2], [3], [4], [5]]

# Children
children = [[] for _ in range(6)]
for j in range(6):
    if parent[j] != -1:
        children[parent[j]].append(j)

print("Elimination tree:")
for j in range(6):
    p = parent[j] if parent[j] != -1 else "root"
    c = children[j] if children[j] else "leaf"
    print(f"  Node {j}: parent={p}, children={c}")

print("\nTree structure:")
print("        5 (root)")
print("       / \\")
print("      3   4")
print("      |")
print("      2")
print("     / \\")
print("    0   1")

## Step-by-Step Multifrontal Factorization

We process the tree **bottom-up** (leaves first, root last).

At each node $j$:
1. **Assemble** the frontal matrix: original entries from $A$ + contribution blocks from children
2. **Partially factor**: eliminate column(s) belonging to this supernode
3. **Extract contribution block**: the Schur complement passed to the parent

In [ ]:
class FrontalMatrix:
    """A dense frontal matrix indexed by global row/column numbers."""
    
    def __init__(self, indices):
        """indices: sorted list of global row/col indices in this front."""
        self.indices = list(indices)
        self.n = len(indices)
        self.data = np.zeros((self.n, self.n))
        # Map: global index -> local index
        self.local = {g: l for l, g in enumerate(indices)}
    
    def add(self, gi, gj, val):
        """Add val at global position (gi, gj)."""
        li, lj = self.local[gi], self.local[gj]
        self.data[li, lj] += val
        if li != lj:
            self.data[lj, li] += val  # symmetric
    
    def add_contribution(self, contrib):
        """Merge a contribution block (extend-add operation)."""
        for li, gi in enumerate(contrib.indices):
            for lj, gj in enumerate(contrib.indices):
                if gi in self.local and gj in self.local:
                    my_i = self.local[gi]
                    my_j = self.local[gj]
                    self.data[my_i, my_j] += contrib.data[li, lj]
    
    def partial_factor(self, n_eliminate):
        """Partially factor: eliminate the first n_eliminate rows/cols.
        
        Returns:
            L_block: the L entries for the eliminated columns
            d_block: the diagonal entries for D
            contribution: FrontalMatrix with the Schur complement
        """
        ne = n_eliminate
        F = self.data.copy()
        
        L_cols = np.zeros((self.n, ne))
        d_vals = np.zeros(ne)
        
        for k in range(ne):
            d_vals[k] = F[k, k]
            L_cols[k, k] = 1.0
            
            if abs(d_vals[k]) < 1e-15:
                continue
            
            for i in range(k + 1, self.n):
                L_cols[i, k] = F[i, k] / d_vals[k]
            
            # Update trailing submatrix (Schur complement)
            for i in range(k + 1, self.n):
                for j in range(k + 1, self.n):
                    F[i, j] -= L_cols[i, k] * d_vals[k] * L_cols[j, k]
        
        # The contribution block is the remaining (n-ne) x (n-ne) submatrix
        remaining_indices = self.indices[ne:]
        contrib = FrontalMatrix(remaining_indices)
        contrib.data = F[ne:, ne:].copy()
        
        return L_cols, d_vals, contrib
    
    def __repr__(self):
        return f"FrontalMatrix(indices={self.indices})\n{self.data}"

In [ ]:
def multifrontal_factor(A, parent, children):
    """Multifrontal LDL^T factorization.
    
    Returns L (full), d (diagonal of D).
    """
    n = A.shape[0]
    L_full = np.eye(n)
    d_full = np.zeros(n)
    contributions = {}  # node -> FrontalMatrix (contribution block)
    
    # Process bottom-up: postorder traversal
    # (for this simple tree, leaves first then internal nodes)
    order = []
    visited = set()
    
    def postorder(j):
        if j in visited:
            return
        visited.add(j)
        for c in children[j]:
            postorder(c)
        order.append(j)
    
    # Find root(s)
    for j in range(n):
        if parent[j] == -1:
            postorder(j)
    
    print(f"Processing order: {order}\n")
    
    for j in order:
        # Determine the index set for this front
        # Includes j and all rows with nonzeros in column j (below diagonal)
        front_indices = {j}
        for i in range(j + 1, n):
            if abs(A[i, j]) > 1e-15:
                front_indices.add(i)
        
        # Also include indices from children's contributions
        for c in children[j]:
            if c in contributions:
                for idx in contributions[c].indices:
                    front_indices.add(idx)
        
        front_indices = sorted(front_indices)
        front = FrontalMatrix(front_indices)
        
        # Step 1: Assemble original entries
        for gi in front_indices:
            for gj in front_indices:
                if gi >= gj and abs(A[gi, gj]) > 1e-15:
                    li, lj = front.local[gi], front.local[gj]
                    front.data[li, lj] = A[gi, gj]
                    if li != lj:
                        front.data[lj, li] = A[gi, gj]
        
        # Step 2: Merge children's contributions (extend-add)
        for c in children[j]:
            if c in contributions:
                front.add_contribution(contributions[c])
                del contributions[c]  # free memory
        
        print(f"Node {j}: front indices = {front_indices}")
        print(f"  Frontal matrix ({front.n}x{front.n}):")
        print(f"{front.data}")
        
        # Step 3: Partial factorization (eliminate column j)
        L_block, d_block, contrib = front.partial_factor(1)
        
        # Store results
        d_full[j] = d_block[0]
        for k, gi in enumerate(front_indices):
            if gi != j:
                L_full[gi, j] = L_block[k, 0]
        
        print(f"  d[{j}] = {d_block[0]:.4f}")
        print(f"  L column {j}: {dict((gi, L_block[k,0]) for k, gi in enumerate(front_indices) if gi != j and abs(L_block[k,0]) > 1e-15)}")
        
        if contrib.n > 0:
            contributions[j] = contrib
            print(f"  Contribution block (indices={contrib.indices}):")
            print(f"{contrib.data}")
        print()
    
    return L_full, d_full


L_mf, d_mf = multifrontal_factor(A, parent, children)

In [ ]:
# Verify: A = L D L^T
D_mf = np.diag(d_mf)
residual = np.linalg.norm(A - L_mf @ D_mf @ L_mf.T)
print(f"||A - L D L^T|| = {residual:.2e}")

print(f"\nL =")
print(L_mf)
print(f"\nd = {d_mf}")

# Solve a system
b = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0])

# Forward: Ly = b
y = np.linalg.solve(L_mf, b)
# Diagonal: Dz = y
z = y / d_mf
# Backward: L^T x = z
x = np.linalg.solve(L_mf.T, z)

print(f"\nSolution: {x}")
print(f"Residual: {np.linalg.norm(A @ x - b):.2e}")
print(f"NumPy:    {np.linalg.solve(A, b)}")

## The Extend-Add Operation

The key operation that makes multifrontal work is **extend-add**: merging a child's contribution block into the parent's frontal matrix.

The child's contribution block is a dense matrix indexed by a subset of the parent's indices. Extend-add:
1. Maps each child index to the corresponding parent index
2. Adds the child's values into the parent's frontal matrix at those positions

This is where fill-in gets **assembled** — it happens during extend-add, not as scattered updates to a global sparse matrix.

In [ ]:
# Illustrate extend-add
print("Example: Node 2 receives contributions from children 0 and 1")
print()

# Child 0's contribution (indices {2})
child0_contrib = FrontalMatrix([2])
child0_contrib.data = np.array([[-0.25]])  # from eliminating column 0
print(f"Child 0 contribution: indices={child0_contrib.indices}")
print(child0_contrib.data)

# Child 1's contribution (indices {2, 4})
child1_contrib = FrontalMatrix([2, 4])
child1_contrib.data = np.array([[-1/3, -1/3], [-1/3, -1/3]])  # from eliminating column 1
print(f"\nChild 1 contribution: indices={child1_contrib.indices}")
print(child1_contrib.data)

# Parent front for node 2 starts with original A entries
parent_front = FrontalMatrix([2, 3, 4])
parent_front.data = np.array([
    [A[2,2], A[2,3], 0],    # A[2,4] = 0
    [A[3,2], A[3,3], A[3,4]],
    [0, A[4,3], A[4,4]]
])
print(f"\nParent front BEFORE extend-add:")
print(parent_front.data)

# Extend-add child contributions
parent_front.add_contribution(child0_contrib)
parent_front.add_contribution(child1_contrib)
print(f"\nParent front AFTER extend-add:")
print(parent_front.data)
print("\nNotice: position (2,2) accumulated contributions from both children")
print("        position (2,4) got a NEW entry from child 1 — this is FILL-IN!")

## Parallelism in the Multifrontal Method

The elimination tree directly reveals parallelism. Nodes on **independent branches** (siblings) can be processed simultaneously.

In our example:
- Nodes 0 and 1 are independent → factor in parallel
- Node 2 depends on 0 and 1 → must wait
- Nodes 3 and 4 are independent → factor in parallel
- Node 5 depends on 3 and 4 → must wait

rmumps uses **rayon** (Rust's parallel library) to process independent level-sets concurrently.

In [ ]:
def compute_levels(parent, children, n):
    """Compute level-sets for parallel processing.
    
    Level 0: leaves (no children)
    Level k: nodes whose children are all in levels < k
    """
    level = np.zeros(n, dtype=int)
    
    def compute_level(j):
        if not children[j]:
            return 0
        return 1 + max(compute_level(c) for c in children[j])
    
    for j in range(n):
        level[j] = compute_level(j)
    
    # Group by level
    max_level = max(level)
    levels = [[] for _ in range(max_level + 1)]
    for j in range(n):
        levels[level[j]].append(j)
    
    return levels


levels = compute_levels(parent, children, 6)
print("Level-sets (nodes at the same level can be processed in parallel):")
for i, level in enumerate(levels):
    parallel_tag = " ← parallel" if len(level) > 1 else ""
    print(f"  Level {i}: {level}{parallel_tag}")

print(f"\nSequential depth: {len(levels)} steps")
print(f"Total nodes: 6")
print(f"Max parallelism: {max(len(l) for l in levels)} nodes at once")

## Scaling: Why Multifrontal Wins

Let's compare approaches on increasingly large problems.

In [ ]:
import time
from scipy import sparse
from scipy.sparse.linalg import splu

def laplacian_2d_scipy(m):
    n = m * m
    rows, cols, vals = [], [], []
    for ix in range(m):
        for iy in range(m):
            row = ix * m + iy
            rows.append(row); cols.append(row); vals.append(4.0)
            if ix > 0:     rows.append(row); cols.append((ix-1)*m+iy); vals.append(-1.0)
            if ix < m-1:   rows.append(row); cols.append((ix+1)*m+iy); vals.append(-1.0)
            if iy > 0:     rows.append(row); cols.append(ix*m+(iy-1)); vals.append(-1.0)
            if iy < m-1:   rows.append(row); cols.append(ix*m+(iy+1)); vals.append(-1.0)
    return sparse.csc_matrix((vals, (rows, cols)), shape=(n, n))


print(f"{'Grid':>6} {'n':>7} {'Dense (s)':>12} {'Sparse (s)':>12} {'Speedup':>8}")
print("-" * 50)

for m in [10, 20, 30, 50, 80]:
    n = m * m
    A_sp = laplacian_2d_scipy(m)
    b = np.random.randn(n)
    
    # Dense solve
    if n <= 2500:  # only time dense for small problems
        A_dense = A_sp.toarray()
        t0 = time.perf_counter()
        x_dense = np.linalg.solve(A_dense, b)
        t_dense = time.perf_counter() - t0
    else:
        t_dense = float('nan')
    
    # Sparse solve (multifrontal via SuperLU)
    t0 = time.perf_counter()
    lu = splu(A_sp)
    x_sparse = lu.solve(b)
    t_sparse = time.perf_counter() - t0
    
    speedup = t_dense / t_sparse if not np.isnan(t_dense) else float('nan')
    dense_str = f"{t_dense:.4f}" if not np.isnan(t_dense) else "(too slow)"
    speedup_str = f"{speedup:.1f}x" if not np.isnan(speedup) else "—"
    
    print(f"{m}x{m:>3} {n:>7} {dense_str:>12} {t_sparse:>12.4f} {speedup_str:>8}")

## Iterative Refinement

After solving $Ax = b$ via $LDL^T$, the solution may have rounding errors. **Iterative refinement** improves accuracy cheaply:

1. Compute residual: $r = b - Ax$
2. Solve for correction: $LDL^T \delta x = r$ (reuses the existing factorization!)
3. Update: $x \leftarrow x + \delta x$
4. Repeat until $\|r\|$ is small enough

Each refinement step costs only $O(n)$ for the residual + $O(\text{nnz}(L))$ for the solve — much cheaper than the $O(\text{nnz}(L))$ factorization.

In [ ]:
def solve_with_refinement(A, L, d, b, max_iters=5):
    """Solve Ax = b with iterative refinement."""
    n = len(b)
    
    # Initial solve
    y = np.linalg.solve(L, b)
    z = y / d
    x = np.linalg.solve(L.T, z)
    
    print(f"{'Iter':>4} {'||r||':>12} {'||dx||':>12}")
    
    for it in range(max_iters):
        r = b - A @ x
        r_norm = np.linalg.norm(r)
        
        if r_norm < 1e-14:
            print(f"{it:>4} {r_norm:>12.2e}  (converged)")
            break
        
        # Solve for correction
        y = np.linalg.solve(L, r)
        z = y / d
        dx = np.linalg.solve(L.T, z)
        
        x += dx
        print(f"{it:>4} {r_norm:>12.2e} {np.linalg.norm(dx):>12.2e}")
    
    return x


# Use our multifrontal factors
b = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0])
D_diag = np.diag(np.diag(D_mf)) if D_mf.ndim == 2 else D_mf
x_refined = solve_with_refinement(A, L_mf, d_mf, b)
print(f"\nFinal residual: {np.linalg.norm(A @ x_refined - b):.2e}")

## The Complete Picture: From Problem to Solution

Here's the full pipeline that rmumps implements:

```
┌─────────────────────────────────────────────────────┐
│  ANALYZE PHASE (once per sparsity pattern)          │
│                                                     │
│  1. COO → CSC conversion                            │
│  2. AMD ordering → permutation P                    │
│  3. Elimination tree from PAP^T                     │
│  4. Supernode detection                             │
│  5. Memory allocation for frontal matrices          │
└─────────────────────┬───────────────────────────────┘
                      │
                      ▼
┌─────────────────────────────────────────────────────┐
│  FACTOR PHASE (once per iteration)                  │
│                                                     │
│  For each supernode (bottom-up in etree):           │
│    a. Assemble frontal matrix from A + children     │
│    b. Bunch-Kaufman pivot selection                 │
│    c. Partial dense LDL^T factorization             │
│    d. Compute contribution block (Schur complement) │
│    e. Extend-add to parent                          │
│                                                     │
│  Independent supernodes factored in parallel        │
│  (rayon thread pool)                                │
└─────────────────────┬───────────────────────────────┘
                      │
                      ▼
┌─────────────────────────────────────────────────────┐
│  SOLVE PHASE (once per right-hand side)             │
│                                                     │
│  1. Apply permutation: b' = Pb                      │
│  2. Forward substitution: Ly = b'                   │
│  3. Block diagonal solve: Dz = y                    │
│  4. Backward substitution: L^T x' = z              │
│  5. Undo permutation: x = P^T x'                   │
│  6. Iterative refinement (adaptive, up to 10 steps) │
└─────────────────────────────────────────────────────┘
```

## The Journey We've Taken

| Note | Topic | Key Idea | Cost |
|------|-------|----------|------|
| 1 | Naive GE | Forward elimination + back substitution | $O(n^3)$, unstable |
| 2 | Pivoting + LU | Row swaps for stability, factor once solve many | $O(n^3)$ factor, $O(n^2)$ solve |
| 3 | LDL^T + Bunch-Kaufman | Exploit symmetry, handle indefiniteness, get inertia | $\frac{1}{3}n^3$, half storage |
| 4 | COO + CSC | Sparse formats, store only nonzeros | $O(\text{nnz})$ storage |
| 5 | AMD + Elimination Trees | Minimize fill-in, structure the factorization | Analyze once, factor many |
| 6 | Multifrontal | Dense ops on frontal matrices, tree parallelism | Near-optimal for sparse |

Each step solved a specific problem:
- **Note 2 fixed Note 1's instability** (pivoting)
- **Note 3 halved Note 2's cost** (symmetry) and added inertia
- **Note 4 broke the $O(n^2)$ storage barrier** (sparsity)
- **Note 5 controlled fill-in** (ordering) and structured the work (elimination tree)
- **Note 6 made everything fast** (dense BLAS on frontal matrices + parallelism)

This is exactly the progression implemented in ripopt:
- Small problems ($n < 110$): dense LDL^T with Bunch-Kaufman (`src/linear_solver/dense.rs`)
- Large problems ($n \geq 110$): multifrontal via rmumps (`rmumps/`)
- Banded problems: auto-detected, $O(n \cdot b^2)$ solver (`src/linear_solver/banded.rs`)

---

*This is Note 6 in a series building from basic Gaussian elimination to the multifrontal sparse solver used in [ripopt](https://github.com/jkitchin/ripopt).*